# 36문항 기본 BM25 LangSmith

3개 사업 × 12문항으로 평가 범위를 확대한 기본 BM25 버전.

실행 출력은 GitHub 공유를 위해 제거했습니다. 원본 문서와 평가 데이터는 별도로 준비해야 합니다.


# RFP RAG: 기본 BM25 / Kiwi BM25 비교

이 노트북은 **첫 3개 사업 × 각 12문항 = 36문항**을 평가합니다. 짝이 되는 다른 노트북과 설치 패키지, 문서 처리, 임베딩, 검색 후보 수, 재정렬, 답변 생성 모델, Claude 평가 기준이 같고 `BM25_MODE`만 다릅니다.

1. Colab에서 GPU 런타임을 선택하고 첫 설치 셀부터 순서대로 실행합니다.
2. Google Drive의 `/content/drive/MyDrive/rag_eval_dataset.json`에 평가셋을 둡니다. 기존 RFP 문서 폴더와 `data_list.csv` 경로도 아래 로드 셀의 설정을 따릅니다.
3. Colab **보안 비밀**에 `LANGSMITH_API_KEY`, `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`를 각각 등록합니다. 키 값은 노트북에 적거나 출력하지 않습니다.
4. 마지막 LangSmith 셀을 실행하면 두 노트북이 동일한 이름의 36문항 데이터셋을 사용하고, 각각 별도 실험을 생성합니다. LangSmith의 데이터셋 화면에서 두 실험을 비교합니다.

터미널 환경에서 실행할 때 사용자가 준 quickstart 명령의 형태는 아래와 같습니다. 노트북 자체는 Colab에서 `%pip`로 패키지를 설치하므로 별도 가상환경을 만들 필요가 없습니다.

```bash
mkdir ls-quickstart && cd ls-quickstart
python -m venv .venv && source .venv/bin/activate
pip install -U langsmith openai anthropic
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY="<your-langsmith-api-key>"
export OPENAI_API_KEY="<your-openai-api-key>"
export ANTHROPIC_API_KEY="<your-claude-api-key>"
```

로컬 Windows PowerShell에서는 활성화 명령과 환경 변수 설정 문법이 다릅니다. Colab에서는 아래 보안 비밀 셀을 사용하세요.

**이 파일의 검색 모드: current**


In [ ]:
%pip install -q numpy pandas faiss-cpu "sentence-transformers==5.6.0" langchain-huggingface langchain-community langchain-text-splitters pymupdf4llm langchain-hwp-hwpx-loader "transformers==4.51.3" accelerate rank_bm25 kiwipiepy
%pip install -U -q langsmith openai anthropic langchain-openai


In [ ]:
import os

def require_secret(name):
    if not os.getenv(name):
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
    if not os.getenv(name):
        raise RuntimeError(f"{name}을(를) Colab 보안 비밀 또는 환경 변수에 설정하세요.")

for secret_name in ("LANGSMITH_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    require_secret(secret_name)
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "rfp-bm25-kiwi-3x12"
print("API 키가 설정되었습니다. 값은 출력하지 않습니다.")


In [ ]:
import os
import re
import unicodedata

import pandas as pd
import pymupdf4llm
from google.colab import drive
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from hwp_hwpx_parser import ExtractOptions, TableStyle
from langchain_text_splitters import RecursiveCharacterTextSplitter

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/data"
metadata_csv = "/content/drive/MyDrive/data_list.csv"


In [ ]:
# 문서 순서를 고정해 다시 실행해도 청크 번호가 같은 순서로 생성되도록 합니다.
sorted_file_names = sorted(
    os.listdir(folder),
    key=lambda name: (unicodedata.normalize("NFKC", name).casefold(), name),
)
print("정렬한 파일 수:", len(sorted_file_names))

## 1. 문서 로드 (HWP, PDF)


In [ ]:
def normalize_filename(name):
    name = str(name).rsplit(".", 1)[0]
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip()


df = pd.read_csv(metadata_csv, encoding="utf-8")
loaded_files = []

for name in sorted_file_names:
    full_path = os.path.join(folder, name)

    extension = name.rsplit(".", 1)[-1].lower()
    if extension == "pdf":
        text = pymupdf4llm.to_markdown(full_path)
    elif extension in ("hwp"):
        options = ExtractOptions(table_style=TableStyle.MARKDOWN)
        loader = HwpHwpxLoader(full_path, mode="single", extract_options=options)
        text = loader.load()[0].page_content

    loaded_files.append((name, text))

print("로드한 문서:", len(loaded_files))


## 2. 메타데이터와 본문을 Document로 저장

`page_content`에 본문을, `metadata`에 CSV의 문서 정보를 저장합니다.


In [ ]:
metadata_columns = [
    "공고 번호", "공고 차수", "사업명", "사업 금액", "발주 기관",
    "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일", "파일명",
]

documents = []
for name, text in loaded_files:
    result_df = df[df["파일명"].apply(normalize_filename) == normalize_filename(name)]
    if result_df.empty:
        continue

    row = result_df.iloc[0]
    metadata = {column: row[column] for column in metadata_columns}
    documents.append(Document(page_content=text, metadata=metadata))

print("메타데이터와 연결한 문서:", len(documents))


## 3. 본문 기본 전처리

연속 공백과 빈 줄만 정리합니다.


In [ ]:
for document in documents:
    text = document.page_content.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    document.page_content = re.sub(r"\n{3,}", "\n\n", text).strip()


## 4. 청킹


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " "], chunk_size=700, chunk_overlap=100
)

chunks = splitter.split_documents(documents)
for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id

print("청크 수:", len(chunks))


## 5. 임베딩 모델 준비

사업별 FAISS 인덱스는 해당 사업의 질문이 처음 들어올 때 만듭니다.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model_name = "dragonkue/BGE-m3-ko" #"dragonkue/BGE-m3-ko" #"nlpai-lab/KURE-v1" #"BAAI/bge-m3" # "Alibaba-NLP/gte-multilingual-base"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cuda", "trust_remote_code": True},
    encode_kwargs={"batch_size": 64, "normalize_embeddings": True},
)

print("임베딩 모델:", embedding_model_name)


## 6. 질문에서 사업 식별 후 하이브리드 검색

질문에 적힌 정식 사업명을 문서 메타데이터의 사업명 목록과 비교합니다. 띄어쓰기와 문장부호 차이는 무시합니다. 줄임말은 `PROJECT_ALIASES`에 등록하세요. 사업을 하나로 특정할 수 없으면 오류를 내며, 평가 데이터의 `project` 필드는 검색에 사용하지 않습니다.


In [ ]:
from collections import defaultdict
from langchain_community.retrievers import BM25Retriever


def normalize_project_text(value):
    return "".join(
        char for char in unicodedata.normalize("NFKC", str(value)).casefold()
        if char.isalnum()
    )


def metadata_text(value):
    return "" if pd.isna(value) else str(value).strip()


# 줄임말을 쓰려면 아래에 "줄임말": ("발주 기관", "정식 사업명")을 추가합니다.
PROJECT_ALIASES = {}

project_chunks_by_key = defaultdict(list)
for doc in chunks:
    agency = metadata_text(doc.metadata.get("발주 기관"))
    title = metadata_text(doc.metadata.get("사업명"))
    if title:
        project_chunks_by_key[(agency, title)].append(doc)

project_catalog = sorted(project_chunks_by_key)
if not project_catalog:
    raise ValueError("사업명 메타데이터가 있는 청크가 없습니다.")
print("식별 가능한 사업:", len(project_catalog))


def choose_project(matches, query):
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0]

    normalized_query = normalize_project_text(query)
    agency_matches = [
        key for key in matches
        if key[0] and normalize_project_text(key[0]) in normalized_query
    ]
    if len(agency_matches) == 1:
        return agency_matches[0]

    labels = ", ".join(f'[{agency}] "{title}"' for agency, title in matches)
    raise ValueError(f"사업을 하나로 특정할 수 없습니다: {labels}")


def project_title_matches_query(project_key, normalized_query):
    agency, title = project_key
    normalized_title = normalize_project_text(title)
    if normalized_title in normalized_query:
        return True

    # 기관명이 사업명 앞에 붙어 있지만 질문에서는 조사와 함께 분리된 경우.
    normalized_agency = normalize_project_text(agency)
    if normalized_agency and normalized_title.startswith(normalized_agency):
        short_title = normalized_title[len(normalized_agency):]
        if len(short_title) >= 8 and normalized_agency in normalized_query:
            return short_title in normalized_query
    return False


def project_identity_matches(project_key, expected_project):
    agency, title = project_key
    expected = normalize_project_text(expected_project)
    return expected in {
        normalize_project_text(title),
        normalize_project_text(agency + title),
    }


def identify_project(query):
    normalized_query = normalize_project_text(query)
    if not normalized_query:
        raise ValueError("질문이 비어 있습니다.")

    # 정식 사업명이 질문에 들어 있으면 포함 관계의 짧은 이름만 제외합니다.
    direct_matches = [
        key for key in project_catalog
        if project_title_matches_query(key, normalized_query)
    ]
    if direct_matches:
        specific_matches = [
            key for key in direct_matches
            if not any(
                normalize_project_text(key[1]) != normalize_project_text(other[1])
                and normalize_project_text(key[1]) in normalize_project_text(other[1])
                for other in direct_matches
            )
        ]
        return choose_project(specific_matches, query)

    # 정식 명칭이 없는 질문은 사용자가 등록한 줄임말만 사용합니다.
    alias_matches = [
        (alias, target) for alias, target in PROJECT_ALIASES.items()
        if normalize_project_text(alias) in normalized_query
    ]
    if alias_matches:
        longest = max(len(normalize_project_text(alias)) for alias, _ in alias_matches)
        targets = [
            target for alias, target in alias_matches
            if len(normalize_project_text(alias)) == longest
        ]
        unknown = [target for target in targets if target not in project_chunks_by_key]
        if unknown:
            raise ValueError(f"PROJECT_ALIASES의 사업이 문서 목록에 없습니다: {unknown}")
        return choose_project(targets, query)

    raise ValueError(
        "질문에서 사업명을 찾지 못했습니다. 정식 사업명을 질문에 적거나 "
        "PROJECT_ALIASES에 줄임말을 등록해 주세요."
    )


from kiwipiepy import Kiwi

kiwi = Kiwi()
BM25_MODES = ("current", "kiwi")
BM25_MODE = "current"
PUNCTUATION_TAGS = {"SF", "SP", "SS", "SE", "SO", "SW"}


def kiwi_morphs(text):
    """Tokenize documents and queries identically for Kiwi BM25."""
    return [
        token.form
        for token in kiwi.tokenize(text)
        if token.form.strip() and token.tag not in PUNCTUATION_TAGS
    ]


dense_retrievers_by_project = {}
bm25_retrievers_by_project_and_mode = {}


def get_retrievers(project_key, bm25_mode):
    if bm25_mode not in BM25_MODES:
        raise ValueError(f"지원하지 않는 BM25 방식: {bm25_mode}")

    project_chunks = project_chunks_by_key[project_key]
    if project_key not in dense_retrievers_by_project:
        store = FAISS.from_documents(project_chunks, embeddings)
        dense_retrievers_by_project[project_key] = store.as_retriever(search_kwargs={"k": 20})

    bm25_key = (project_key, bm25_mode)
    if bm25_key not in bm25_retrievers_by_project_and_mode:
        if bm25_mode == "kiwi":
            bm25 = BM25Retriever.from_documents(
                project_chunks, preprocess_func=kiwi_morphs
            )
        else:
            bm25 = BM25Retriever.from_documents(project_chunks)
        bm25.k = 20
        bm25_retrievers_by_project_and_mode[bm25_key] = bm25

    return (
        dense_retrievers_by_project[project_key],
        bm25_retrievers_by_project_and_mode[bm25_key],
    )


def rrf_search(query, project_key, bm25_mode, k=60):
    dense_retriever, bm25_retriever = get_retrievers(project_key, bm25_mode)
    dense_results = dense_retriever.invoke(query)
    sparse_results = bm25_retriever.invoke(query)
    scores = {}

    for results in (dense_results, sparse_results):
        for rank, doc in enumerate(results, start=1):
            doc_id = doc.metadata["chunk_id"]
            if doc_id not in scores:
                scores[doc_id] = {"doc": doc, "score": 0}
            scores[doc_id]["score"] += 1 / (k + rank)

    ranked = sorted(scores.values(), key=lambda item: item["score"], reverse=True)
    return [item["doc"] for item in ranked]


## 7. HyDE로 검색 문장 만들기


In [ ]:
import gc
import torch

"del embeddings"

gc.collect()
torch.cuda.empty_cache()

In [ ]:
hyde_search_rules = '''너는 RFP에서 관련 문단을 찾기 위한 가상 안내문 작성자야.
실제 답변이 아니라 검색에 쓸 안내문 1문장만 작성해.
- 질문의 핵심 주제어를 그대로 포함해.
- 질문과 관계없는 다른 사업이나 항목으로 이동하지 마.
- 확인되지 않은 금액, 날짜, 법률, 자격 조건을 단정하지 마.
- 제목, 목록, 따옴표, 추가 설명을 쓰지 마.

예시:
질문: 이 사업의 입찰 참여 자격은 무엇인가요?
가상 안내문: 이 사업의 입찰 참여 자격에 관한 조건이 안내되어 있다.'''

In [ ]:
# 이 비교에서는 아래 셀의 GPT-5-mini를 사용합니다.
# 기존 EXAONE AWQ 로딩은 실행하지 않습니다.
method = "basic"


In [ ]:
import os
from langchain_openai import ChatOpenAI

method = "basic"  # "basic" 또는 "hyde"

# Colab 보안 비밀에 키를 등록한 경우
if not os.getenv("OPENAI_API_KEY"):
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

chat_model = ChatOpenAI(model="gpt-5-mini-2025-08-07")

In [ ]:
def make_search_text(query, method):
    if method == "basic":
        return query
    if method == "hyde":
        hyde_prompt = f"{hyde_search_rules}\n\n질문: {query}\n가상 안내문:"
        return chat_model.invoke(hyde_prompt).content.strip()
    raise ValueError(f"지원하지 않는 검색 방식: {method}")

## 8. 질문에서 식별한 사업 안에서 재순위화


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "Alibaba-NLP/gte-multilingual-reranker-base",
    trust_remote_code=True,
    device="cuda",
)

# 질문과 검색 설정이 같을 때만 검색 결과를 재사용합니다.
retrieval_cache = {}


def retrieve_and_rerank(query, method, bm25_mode=None):
    bm25_mode = BM25_MODE if bm25_mode is None else bm25_mode
    project_key = identify_project(query)
    cache_key = (project_key, method, query, bm25_mode)
    if cache_key not in retrieval_cache:
        search_text = make_search_text(query, method)
        found = rrf_search(search_text, project_key, bm25_mode)
        ranking = reranker.rank(
            query,
            [doc.page_content for doc in found],
            top_k=6,
        )
        reranked_docs = [found[row["corpus_id"]] for row in ranking]
        retrieval_cache[cache_key] = (
            project_key, search_text, found, reranked_docs
        )
    return retrieval_cache[cache_key]


## 9. 일반 질문 함수

아래 함수 셀까지 실행하면 `answer_question("정식 사업명의 공고기간은?")`처럼 평가셋 밖의 질문에도 사용할 수 있습니다. 정식 명칭을 줄여 부르려면 검색 셀의 `PROJECT_ALIASES`에 등록하세요.


In [ ]:
import gc
import torch

"del embeddings"

gc.collect()
torch.cuda.empty_cache()

In [ ]:
answer_rules = '''
너는 RFP 문서의 사실 확인 담당자다. 아래 <문서>에 적힌 내용만으로 <질문>에 답한다.

규칙:
1. 질문이 여러 항목을 묻는다면 항목별로 근거를 따로 찾는다.
2. 질문의 사업명과 다른 사업의 정보는 사용하지 않는다.
3. 각 항목에 대해 문서에서 직접 인용할 수 있는 문구를 먼저 제시한다.
4. 직접 인용할 근거가 없는 항목은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 쓴다.
5. 금액, 기간, 날짜, 비율, 인력, 제출 조건은 문서의 표현을 그대로 옮긴다. 계산하거나 일반적인 관행으로 빈칸을 채우지 않는다.
6. 답변의 각 사실 바로 뒤에 근거 문서 번호 [1]처럼 붙인다.
7. HyDE 검색 문장과 모델의 사전 지식은 근거로 사용하지 않는다.

출력 형식:
근거:
- 항목명: [문서 번호] "문서에서 그대로 가져온 짧은 문구"
- 근거가 없는 항목명: 확인 불가

답변:
근거가 확인된 내용만 간결하게 답한다. 근거가 없는 항목은 확인할 수 없다고 밝힌다.

<문서>
{context}
</문서>

<질문>
{query}
</질문>
'''


def answer_question(query, search_method=None, bm25_mode=None):
    bm25_mode = BM25_MODE if bm25_mode is None else bm25_mode
    selected_method = method if search_method is None else search_method
    project_key, search_text, found, reranked_docs = retrieve_and_rerank(
        query, selected_method, bm25_mode
    )
    context = "\n\n".join(
        f"[{j}] 메타데이터: {doc.metadata}\n본문: {doc.page_content}"
        for j, doc in enumerate(reranked_docs, start=1)
    )
    answer = chat_model.invoke(
        answer_rules.format(context=context, query=query)
    ).content.strip()
    return {
        "query": query,
        "bm25_mode": bm25_mode,
        "identified_agency": project_key[0],
        "identified_project": project_key[1],
        "search_text": search_text,
        "answer": answer,
        "contexts": [doc.page_content for doc in reranked_docs],
        "chunk_ids": [doc.metadata["chunk_id"] for doc in reranked_docs],
        "rrf_chunk_ids": [doc.metadata["chunk_id"] for doc in found],
    }



### 평가 질문 불러오기 (선택)

평가셋을 사용할 때만 실행합니다. 일반 질문의 사업 식별에는 평가 데이터의 `project` 값을 사용하지 않습니다.


In [ ]:
import json
from collections import Counter
from pathlib import Path

eval_dataset_candidates = [
    Path("/content/drive/MyDrive/rag_eval_dataset.json"),
    Path("rag_eval_dataset.json"),
]
eval_dataset_path = next((p for p in eval_dataset_candidates if p.is_file()), None)
if eval_dataset_path is None:
    raise FileNotFoundError("rag_eval_dataset.json을 Google Drive 루트 또는 노트북 폴더에 두세요.")

with eval_dataset_path.open(encoding="utf-8") as file:
    all_eval_records = json.load(file)

EVAL_PROJECT_COUNT = 3
EVAL_QUESTIONS_PER_PROJECT = 12
available_projects = list(dict.fromkeys(r["project"] for r in all_eval_records))
selected_projects = available_projects[:EVAL_PROJECT_COUNT]
selected_eval_records = []
selected_count = Counter()
for record in all_eval_records:
    project = record["project"]
    if project in selected_projects and selected_count[project] < EVAL_QUESTIONS_PER_PROJECT:
        selected_eval_records.append(record)
        selected_count[project] += 1

if len(selected_projects) != 3 or len(selected_eval_records) != 36 or any(
    selected_count[p] != 12 for p in selected_projects
):
    raise ValueError(f"3개 사업에서 각 12문항을 선택하지 못했습니다: {dict(selected_count)}")

evaluation_data = []
for record in selected_eval_records:
    if not all(record.get(field) for field in ("id", "question", "reference", "project")):
        raise ValueError(f"필수 평가 필드가 비어 있습니다: {record.get('id')}")
    evaluation_data.append({
        "id": record["id"],
        "query": record["question"],
        "reference": record["reference"],
        "project": record["project"],
        "question_type": record.get("question_type", ""),
        "evidence_quotes": record.get("evidence_quotes", []),
    })

if len({r["id"] for r in evaluation_data}) != 36:
    raise ValueError("평가 문항 ID가 중복됩니다.")
if len({r["query"] for r in evaluation_data}) != 36:
    raise ValueError("평가 질문이 중복됩니다.")

question = evaluation_data
print(f"평가셋: {eval_dataset_path} / {len(evaluation_data)}문항")
print("사업별:", dict(Counter(r["project"] for r in evaluation_data)))
print("유형별:", dict(Counter(r["question_type"] for r in evaluation_data)))


### LangSmith 데이터셋

두 파일은 동일한 36문항 데이터셋을 공유합니다.


In [ ]:
import hashlib
from langsmith import Client

# 두 노트북이 동일한 평가셋을 공유합니다. 질문/정답이 바뀌면 해시도 바뀝니다.
dataset_payload = json.dumps(evaluation_data, ensure_ascii=False, sort_keys=True)
dataset_hash = hashlib.sha256(dataset_payload.encode("utf-8")).hexdigest()[:12]
DATASET_NAME = f"rfp-bm25-3x12-{dataset_hash}"
ls_client = Client()

if ls_client.has_dataset(dataset_name=DATASET_NAME):
    dataset = ls_client.read_dataset(dataset_name=DATASET_NAME)
    existing = list(ls_client.list_examples(dataset_id=dataset.id))
    expected = {r["id"]: r for r in evaluation_data}
    if len(existing) != 36 or set(x.inputs.get("id") for x in existing) != set(expected):
        raise RuntimeError("기존 LangSmith 데이터셋의 36문항이 현재 평가셋과 다릅니다.")
    for example in existing:
        item = expected[example.inputs["id"]]
        if example.inputs.get("question") != item["query"] or (example.outputs or {}).get("reference") != item["reference"]:
            raise RuntimeError(f"LangSmith 데이터셋 내용이 다릅니다: {item['id']}")
else:
    dataset = ls_client.create_dataset(
        dataset_name=DATASET_NAME,
        description="동일한 첫 3개 사업의 각 12문항으로 기본 BM25와 Kiwi BM25를 비교",
    )
    ls_client.create_examples(
        dataset_id=dataset.id,
        examples=[{
            "inputs": {"id": r["id"], "question": r["query"], "project": r["project"], "question_type": r["question_type"]},
            "outputs": {"reference": r["reference"], "evidence_quotes": r["evidence_quotes"]},
        } for r in evaluation_data],
    )

print("LangSmith 데이터셋:", DATASET_NAME, "/ 36문항")


### LangSmith 평가

OpenAI로 답변을 생성하고 Claude가 기준 답변 대비 0~1로 채점합니다. 근거 인용 구절의 정확 일치 포함률과 사업 식별도 함께 기록합니다. Claude 점수는 RAGAS 점수와 다른 척도입니다.


In [ ]:
import json
import re
import unicodedata
import anthropic

CLAUDE_JUDGE_MODEL = os.getenv("CLAUDE_JUDGE_MODEL", "claude-sonnet-4-6")
claude_client = anthropic.Anthropic()

def langsmith_target(inputs: dict) -> dict:
    # 답변 생성 전에 사업을 검사하고, 기관명은 검색기의 사업 키에서 가져옵니다.
    identified_key = identify_project(inputs["question"])
    if not project_identity_matches(identified_key, inputs["project"]):
        raise ValueError(
            f"사업 식별 오류: {inputs['id']} / "
            f"식별 결과={identified_key!r} / 평가셋={inputs['project']!r}"
        )
    record = answer_question(inputs["question"], bm25_mode=BM25_MODE)
    return {
        "answer": record["answer"].split("답변:", 1)[-1].strip(),
        "full_answer": record["answer"],
        "contexts": record["contexts"],
        "chunk_ids": record["chunk_ids"],
        "rrf_chunk_ids": record["rrf_chunk_ids"],
        "identified_project": identified_key[1],
        "identified_agency": identified_key[0],
        "bm25_mode": BM25_MODE,
    }

def claude_answer_score(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    if not outputs or "answer" not in outputs:
        return {"key": "claude_answer_score", "score": None, "comment": "답변 생성 실패"}
    prompt = (
        "질문, 기준 답변, 생성 답변을 비교해 사실 정확성과 필수 항목의 충족 정도를 0~1로 평가하세요. "
        "추가 설명이 사실과 충돌하면 감점하세요. 기준 답변에 없는 사소한 표현 차이는 감점하지 마세요. "
        "반드시 JSON 객체 하나만 반환하세요: {\"score\": 0.0, \"reason\": \"한 문장 근거\"}.\n\n"
        f"질문: {inputs['question']}\n\n기준 답변: {reference_outputs['reference']}\n\n생성 답변: {outputs['answer']}"
    )
    message = claude_client.messages.create(
        model=CLAUDE_JUDGE_MODEL,
        max_tokens=256,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    judge_text = "".join(block.text for block in message.content if block.type == "text")
    match = re.search(r"\{.*\}", judge_text, flags=re.S)
    if match is None:
        raise ValueError(f"Claude 평가 JSON을 찾지 못했습니다: {judge_text[:200]}")
    graded = json.loads(match.group(0))
    score = float(graded["score"])
    if not 0 <= score <= 1:
        raise ValueError(f"Claude 점수가 0~1 범위를 벗어났습니다: {score}")
    return {"key": "claude_answer_score", "score": score, "comment": str(graded.get("reason", ""))}

def compact(text):
    return re.sub(r"\s+", "", unicodedata.normalize("NFC", str(text)))

def evidence_quote_coverage(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    if not outputs or "contexts" not in outputs:
        return {"key": "exact_quote_coverage", "score": None, "comment": "답변 생성 실패"}
    quotes = []
    for evidence in reference_outputs.get("evidence_quotes", []) or []:
        for curved, straight in re.findall(r'“([^”]+)”|"([^"]+)"', str(evidence)):
            quote = curved or straight
            if "…" in quote or "..." in quote:
                continue
            normalized = compact(quote)
            if len(normalized) >= 8 and re.search(r"[가-힣A-Za-z]", normalized):
                if normalized not in quotes:
                    quotes.append(normalized)
    contexts = [compact(c) for c in outputs["contexts"]]
    found = sum(any(q in c for c in contexts) for q in quotes)
    return {
        "key": "exact_quote_coverage",
        "score": found / len(quotes) if quotes else None,
        "comment": f"발견 {found}/{len(quotes)}; 인용 구절 0개이면 점수 없음",
    }

def correct_project(inputs: dict, outputs: dict) -> dict:
    if not outputs or "identified_project" not in outputs or "identified_agency" not in outputs:
        return {"key": "correct_project", "score": None, "comment": "답변 생성 실패"}
    identified_key = (outputs["identified_agency"], outputs["identified_project"])
    return {
        "key": "correct_project",
        "score": project_identity_matches(identified_key, inputs["project"]),
    }


In [ ]:
# API 호출 전에 36문항의 사업 식별을 전부 검사합니다.
project_check_errors = []
for item in evaluation_data:
    try:
        identified_key = identify_project(item["query"])
        if not project_identity_matches(identified_key, item["project"]):
            project_check_errors.append(f"{item['id']}: {identified_key!r} != {item['project']!r}")
    except ValueError as exc:
        project_check_errors.append(f"{item['id']}: {exc}")
if project_check_errors:
    raise ValueError("평가 전 사업 식별 검사 실패:\n" + "\n".join(project_check_errors))
print("사업 식별 사전 검사: 36/36 통과")

# 이 셀은 OpenAI 답변 생성 36회와 Claude 판정 36회를 수행하고 LangSmith에 저장합니다.
# 두 노트북에서 같은 DATASET_NAME을 확인한 뒤 각각 한 번씩 실행하세요.
results = ls_client.evaluate(
    langsmith_target,
    data=DATASET_NAME,
    evaluators=[claude_answer_score, evidence_quote_coverage, correct_project],
    experiment_prefix=f"rfp-3x12-{BM25_MODE}-claude",
    description="동일한 36문항과 RAG 설정에서 BM25 토큰화만 변경",
    metadata={
        "bm25_mode": BM25_MODE,
        "embedding_model": embedding_model_name,
        "generation_model": "gpt-5-mini-2025-08-07",
        "judge_model": CLAUDE_JUDGE_MODEL,
        "retrieval_top_k": 20,
        "final_context_count": 6,
        "dataset_hash": dataset_hash,
    },
    max_concurrency=1,
    blocking=True,
    error_handling="log",
)
print("LangSmith 실험:", results.url)
print("실험 이름:", results.experiment_name)
result_frame = results.to_pandas()
if len(result_frame) != 36:
    raise RuntimeError(f"평가 결과가 36행이 아닙니다: {len(result_frame)}")
if result_frame["error"].notna().any():
    raise RuntimeError("실패한 답변 생성 실행이 있습니다. LangSmith 실험을 확인하세요.")
if result_frame["feedback.claude_answer_score"].isna().any():
    raise RuntimeError("Claude 판정이 누락된 문항이 있습니다. LangSmith 실험을 확인하세요.")
if not result_frame["feedback.correct_project"].all():
    raise RuntimeError("사업 식별에 실패한 문항이 있습니다.")
print("평가 결과:", len(result_frame), "문항")
print("Claude 평균 점수:", result_frame["feedback.claude_answer_score"].mean())
print("근거 포함률 평균 (인용 구절이 있는 문항):", result_frame["feedback.exact_quote_coverage"].mean())
score_path = eval_dataset_path.with_name(f"langsmith_{BM25_MODE}_3x12_scores.csv")
result_frame[[
    "inputs.id", "inputs.project", "feedback.claude_answer_score",
    "feedback.exact_quote_coverage", "feedback.correct_project",
]].to_csv(score_path, index=False, encoding="utf-8-sig")
print("요약 CSV:", score_path)
